# Дообучение E5-base на silver+gold датасете (Google Colab)

## Что нужно сделать перед запуском

### 1. Выбрать GPU-рантайм
**Runtime → Change runtime type → GPU** (T4 достаточно, но если доступен A100/L4 — возьми его)

### 2. Загрузить данные в Google Drive

На Google Drive внутри папки `thesis` должны лежать **два parquet-файла**:

```
Google Drive/
└── thesis/
    └── data/
        ├── silver/
        │   └── train_gold_silver.parquet     ← train (gold_train + silver_balanced)
        └── golden/
            └── golden_eval.parquet            ← eval (20% golden)
```

Это те же самые файлы, что лежат у тебя локально в `thesis/data/silver/` и `thesis/data/golden/`.
Просто перенеси их в одноимённые папки на Drive.

### 3. Куда попадёт результат

Обученная модель будет заархивирована и сохранена на Drive:

```
Google Drive/
└── thesis/
    └── models/
        └── bi-encoder-e5-finetuned.tar.gz    ← готово к скачиванию
```

Локально (на Colab-машине) модель пишется в `/content/e5_training` ради скорости,
потом архивируется и копируется на Drive в самом конце.

### 4. Что делать локально после обучения

С Google Drive забираешь `thesis/models/bi-encoder-e5-finetuned.tar.gz` и распаковываешь в `thesis/models/`:

```
cd thesis/models/
tar -xzf /path/to/bi-encoder-e5-finetuned.tar.gz
```

Появится папка `thesis/models/bi-encoder-e5-finetuned/`. После этого она загружается
как обычный `SentenceTransformer` из локального пути.


## 1. Установка зависимостей и монтирование Google Drive

In [ ]:
# Установка зависимостей и монтирование Google Drive
#
# ВАЖНО: флаг --upgrade-strategy only-if-needed запрещает pip'у трогать torch.
# Иначе pip подтянет свежий CPU-only torch и GPU работать перестанет.
!pip install -q --upgrade-strategy only-if-needed sentence-transformers datasets pyarrow

from google.colab import drive
drive.mount('/content/drive')

import torch
print(f"torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Жёсткая проверка: если CUDA недоступна — дальше нет смысла
assert torch.cuda.is_available(), (
    "CUDA недоступен. Причины:\n"
    "  1) Не выбран GPU-рантайм: Runtime → Change runtime type → GPU.\n"
    "  2) pip установил CPU-only torch поверх родного. Runtime → Disconnect "
    "and delete runtime, затем запусти ноутбук заново."
)
print(f"Устройство: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## 2. Пути и проверка файлов

In [ ]:
import os
import shutil

# Базовый путь на Google Drive
DRIVE_BASE = "/content/drive/MyDrive/thesis"

# Входные parquet-файлы
train_parquet_path = os.path.join(DRIVE_BASE, "data/silver/train_gold_silver.parquet")
eval_parquet_path  = os.path.join(DRIVE_BASE, "data/golden/golden_eval.parquet")

# Папка для результата на Drive (сюда положим архив модели)
drive_models_dir = os.path.join(DRIVE_BASE, "models")
os.makedirs(drive_models_dir, exist_ok=True)

# Локальные пути на Colab-машине — тренируем здесь (быстрые I/O)
LOCAL_OUTPUT_DIR       = "/content/e5_training"          # чекпоинты во время обучения
LOCAL_FINAL_MODEL_DIR  = "/content/bi-encoder-e5-finetuned"  # финальная модель

# Проверка входных файлов
assert os.path.exists(train_parquet_path), f"Файл не найден: {train_parquet_path}"
assert os.path.exists(eval_parquet_path),  f"Файл не найден: {eval_parquet_path}"

print(f"Train: {train_parquet_path}")
print(f"  размер: {os.path.getsize(train_parquet_path) / 1e6:.1f} MB")
print(f"Eval:  {eval_parquet_path}")
print(f"  размер: {os.path.getsize(eval_parquet_path) / 1e6:.1f} MB")
print(f"Папка для результата: {drive_models_dir}")


## 3. Параметры обучения

In [ ]:
# ==================== ПАРАМЕТРЫ ====================
BASE_MODEL = "intfloat/multilingual-e5-base"   # 768d, zero-shot из HF

BATCH_SIZE       = 32      # E5-base на T4 — 32 безопасно; на A100/L4 можно 64
LEARNING_RATE    = 2e-5
NUM_EPOCHS       = 4
WARMUP_RATIO     = 0.1
EVAL_STEPS       = 500
SAVE_STEPS       = 500
SAVE_TOTAL_LIMIT = 3
LOGGING_STEPS    = 50
SEED             = 42

# E5 требует префиксы: 'query: ' для запросов, 'passage: ' для документов.
# В наших парах product_desc — это query, post_text — passage.
USE_E5_PREFIX = True


## 4. Загрузка и препроцессинг датасетов

In [ ]:
from datasets import load_dataset

train_dataset = load_dataset("parquet", data_files=train_parquet_path, split="train")
eval_dataset  = load_dataset("parquet", data_files=eval_parquet_path,  split="train")

print(f"Train: {len(train_dataset):,} пар, колонки: {train_dataset.column_names}")
print(f"Eval:  {len(eval_dataset):,} пар, колонки: {eval_dataset.column_names}")
print()
print("Пример train:")
print(train_dataset[0])


In [ ]:
# Добавляем E5-префиксы к product_desc и post_text
# (если USE_E5_PREFIX=False, пропускаем этот шаг)

eval_col1 = "product_desc" if "product_desc" in train_dataset.column_names else "description"
print(f"Колонка запроса: {eval_col1}")

if USE_E5_PREFIX:
    def add_prefixes(example):
        return {
            eval_col1:   "query: "   + example[eval_col1],
            "post_text": "passage: " + example["post_text"],
        }

    train_dataset = train_dataset.map(add_prefixes, desc="prefix train")
    eval_dataset  = eval_dataset.map(add_prefixes,  desc="prefix eval")

    print()
    print("После добавления префиксов:")
    print(f"  train[0].{eval_col1}: {train_dataset[0][eval_col1][:120]}...")
    print(f"  train[0].post_text:   {train_dataset[0]['post_text'][:120]}...")
else:
    print("Префиксы НЕ добавлены (USE_E5_PREFIX=False)")


## 5. Модель и loss

In [ ]:
# Совместимость sentence-transformers с новыми transformers
import transformers
from transformers.modeling_utils import PreTrainedModel
transformers.PreTrainedModel = PreTrainedModel

from sentence_transformers import SentenceTransformer
from sentence_transformers.losses import CoSENTLoss

model = SentenceTransformer(BASE_MODEL, device="cuda")

print(f"Модель: {BASE_MODEL}")
print(f"Max seq length: {model.max_seq_length}")
print(f"Embedding dim: {model.get_sentence_embedding_dimension()}")
print(f"Similarity: {model.similarity_fn_name}")

loss = CoSENTLoss(model)
print(f"\nLoss: CoSENTLoss")


## 6. Baseline — Spearman на golden до обучения

In [ ]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator, SimilarityFunction

evaluator = EmbeddingSimilarityEvaluator(
    sentences1=eval_dataset[eval_col1],
    sentences2=eval_dataset["post_text"],
    scores=eval_dataset["score"],
    batch_size=BATCH_SIZE,
    main_similarity=SimilarityFunction.COSINE,
    name="golden",
    show_progress_bar=True,
)

print("Baseline (до обучения):")
baseline_results = evaluator(model)
baseline_spearman = baseline_results["golden_spearman_cosine"]
print(f"  Spearman (cosine): {baseline_spearman:.4f}")
print(f"  Pearson  (cosine): {baseline_results['golden_pearson_cosine']:.4f}")


## 7. Training arguments и запуск

In [ ]:
from sentence_transformers import SentenceTransformerTrainingArguments

args = SentenceTransformerTrainingArguments(
    output_dir=LOCAL_OUTPUT_DIR,

    # Длительность
    num_train_epochs=NUM_EPOCHS,

    # Батч
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,

    # Оптимизатор
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=0.01,

    # Память (на T4 с E5-base это критично)
    gradient_checkpointing=True,
    dataloader_num_workers=2,
    fp16=True,

    # Eval
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    load_best_model_at_end=True,
    metric_for_best_model="eval_golden_spearman_cosine",
    greater_is_better=True,

    # Checkpoints
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,

    # Логирование
    logging_steps=LOGGING_STEPS,

    # Воспроизводимость
    seed=SEED,
    data_seed=SEED,

    dataloader_drop_last=True,
)

total_steps = (len(train_dataset) // BATCH_SIZE) * NUM_EPOCHS
print(f"Ожидаемое кол-во шагов: ~{total_steps:,}")
print(f"Eval каждые {EVAL_STEPS} шагов = ~{total_steps // EVAL_STEPS} раз за обучение")


In [ ]:
from sentence_transformers import SentenceTransformerTrainer

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=loss,
    evaluator=evaluator,
)

# Автоматический resume: если в LOCAL_OUTPUT_DIR есть checkpoint — продолжить с него
has_checkpoint = (
    os.path.isdir(LOCAL_OUTPUT_DIR)
    and any(d.startswith("checkpoint-") for d in os.listdir(LOCAL_OUTPUT_DIR))
)

if has_checkpoint:
    print("Найден checkpoint, продолжаем обучение...")
else:
    print("Checkpoint не найден, начинаем с нуля...")

trainer.train(resume_from_checkpoint=has_checkpoint if has_checkpoint else None)


## 8. Сохранение финальной модели

In [ ]:
# Локально
model.save_pretrained(LOCAL_FINAL_MODEL_DIR)
print(f"Модель сохранена локально: {LOCAL_FINAL_MODEL_DIR}")
print(f"  размер: {sum(os.path.getsize(os.path.join(r,f)) for r,_,fs in os.walk(LOCAL_FINAL_MODEL_DIR) for f in fs) / 1e6:.1f} MB")


## 9. Финальная оценка

In [ ]:
print("Финальная оценка на golden dataset:")
final_results = evaluator(model)
final_spearman = final_results["golden_spearman_cosine"]
final_pearson = final_results["golden_pearson_cosine"]

print(f"\n{'='*50}")
print(f"{'РЕЗУЛЬТАТЫ (E5-base)':^50}")
print(f"{'='*50}")
print(f"{'Метрика':<25} {'До':>10} {'После':>10} {'Delta':>10}")
print(f"{'-'*50}")
print(f"{'Spearman (cosine)':<25} {baseline_spearman:>10.4f} {final_spearman:>10.4f} {final_spearman - baseline_spearman:>+10.4f}")
print(f"{'Pearson (cosine)':<25} {baseline_results['golden_pearson_cosine']:>10.4f} {final_pearson:>10.4f} {final_pearson - baseline_results['golden_pearson_cosine']:>+10.4f}")
print(f"{'='*50}")


## 10. Архивирование и копирование на Google Drive

In [ ]:
# Архивируем локальную папку с моделью и копируем .tar.gz на Drive
ARCHIVE_NAME = "bi-encoder-e5-finetuned.tar.gz"
local_archive = f"/content/{ARCHIVE_NAME}"

assert os.path.exists(LOCAL_FINAL_MODEL_DIR), f"Финальная модель не найдена: {LOCAL_FINAL_MODEL_DIR}"

# tar без лишней обёртки: после распаковки получится сразу bi-encoder-e5-finetuned/
parent_dir = os.path.dirname(LOCAL_FINAL_MODEL_DIR)
base_name  = os.path.basename(LOCAL_FINAL_MODEL_DIR)

!tar -czf {local_archive} -C {parent_dir} {base_name}

archive_size_mb = os.path.getsize(local_archive) / 1e6
print(f"Архив создан: {local_archive} ({archive_size_mb:.1f} MB)")

# Копируем на Google Drive
drive_archive = os.path.join(drive_models_dir, ARCHIVE_NAME)
shutil.copy(local_archive, drive_archive)

assert os.path.exists(drive_archive), "Не удалось скопировать архив на Drive"
print(f"✓ Архив сохранён на Drive: {drive_archive}")
print(f"  Размер: {os.path.getsize(drive_archive) / 1e6:.1f} MB")


## Что делать локально после скачивания

С Google Drive забираешь `thesis/models/bi-encoder-e5-finetuned.tar.gz` и распаковываешь в `thesis/models/`:

```bash
cd thesis/models/
tar -xzf /path/to/bi-encoder-e5-finetuned.tar.gz
```

Появится `thesis/models/bi-encoder-e5-finetuned/`. Дальше её можно использовать как обычный SentenceTransformer:

```python
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("models/bi-encoder-e5-finetuned")
```

**Важно:** E5 всё ещё требует префиксы при инференсе: `'query: ' + query` для запросов и `'passage: ' + text` для документов — точно так же, как было при zero-shot. Дообучение этого не отменяет.

После этого можно:
1. Пересобрать LanceDB таблицу на дообученной E5 (новый `create-lancedb-e5-finetuned.ipynb` по аналогии с `create-lancedb-e5-colab.ipynb`, только `BI_ENCODER_PATH = "models/bi-encoder-e5-finetuned"` и `TABLE_NAME = "e5-base-fine-tuned-50k"` или как-то так).
2. Прогнать `test-bi-encoder.ipynb` с новой таблицей и сравнить с текущей `posts3` (E5 zero-shot) — будет честное сравнение «до/после» дообучения E5.
